# Task 3.4 | Credit risk

> **Credit:** Is credit quality deteriorating? Is cutting limits by 30% for everyone the right fix? If not, propose something better.

**September’s observed default rate was higher.** Among loans that reached 30 days past due (DPD30), the default rate rose from **6.54% in July** to **9.63% in September**. September also had the highest rate when comparing loans disbursed on days 1–22 of each month. The mix of borrowers changed over the same period, so the increase alone is insufficient to conclude that credit quality deteriorated across the portfolio.

| Disbursal month | Short of full repayment at extract | Mature DPD30 default rate | Mature DPD30 rate for disbursals on days 1–22 |
|---|---:|---:|---:|
| July | 17 / 260 (6.54%) | 17 / 260 (6.54%) | 5.76% |
| August | 25 / 427 (5.85%) | 25 / 427 (5.85%) | 5.56% |
| September | 73 / 623 (11.72%) | 34 / 353 (9.63%) | 9.63% |

Finance’s **~12%** is consistent with the share of all September loans still short of full repayment on November 5. That denominator includes **270 loans** that had yet to reach DPD30. The **9.63%** uses the **353 loans** that had reached the checkpoint.

## Where defaults and credit losses concentrate

Meta and TikTok’s combined share of mature loans rose from **26% in July (68/260)** to **41% in September (145/353)**. Across Q3, those channels accounted for **50 of 76 defaults** and **EGP 58,261.27 of EGP 89,831.40 in credit losses**. This makes borrower mix and underwriting in those cohorts the first areas to investigate.

| Acquisition channel | Mature loans | DPD30 defaults | DPD30 rate | Collected fees (EGP) | Credit loss (EGP) | Fees less credit loss (EGP) |
|---|---:|---:|---:|---:|---:|---:|
| Meta | 270 | 41 | 15.19% | 23,800.00 | 46,032.82 | −22,232.82 |
| TikTok | 71 | 9 | 12.68% | 6,760.00 | 12,228.45 | −5,468.45 |
| Unknown | 105 | 4 | 3.81% | 10,800.00 | 5,576.39 | 5,223.61 |
| Other (organic, direct, referral) | 281 | 9 | 3.20% | 26,640.00 | 12,744.88 | 13,895.12 |
| Google Ads | 313 | 13 | 4.15% | 31,360.00 | 13,248.86 | 18,111.14 |
| **Total** | **1,040** | **76** | **7.31%** | **99,360.00** | **89,831.40** | **9,528.60** |

These are observed outcomes by acquisition channel. Fees less credit loss excludes funding, servicing, and collection costs.

## Recommendation

**Hold off on the blanket 30% cut.** Start by reviewing whether borrower profiles, approval criteria, or assigned limits differed in the Meta and TikTok cohorts. The extract records the principal borrowers took out. To evaluate a change in maximum limits, we need the limits they were actually assigned and the amounts they requested.

Use the repayment bands below as a starting point for a targeted policy pilot. The Q3 extract contains one loan per borrower, so repeat repayment history is needed to assess consistency.

| Repayment outcome among 1,040 mature loans | Count | Proposed action |
|---|---:|---|
| Fully repaid by the due date | 658 | Consider quarterly limit increases for borrowers with a consistent record of on-time repayment and verified affordability. |
| Fully repaid after the due date, by DPD30 | 306 | Hold the current limit and review why repayment was late before considering an increase. |
| Defaulted at DPD30 and later repaid in full | 0 | Review cure reasons and test a lower renewal limit once enough cases are available. |
| Defaulted at DPD30 and still short at extract | 76 | Pause new borrowing pending full repayment and reassessment. |

**Data and pilot.** Request assigned limits, requested amounts, underwriting inputs and approval decisions, repeat loan histories, and recorded reasons for late repayment. Once the main risk differences are understood, test targeted changes against the current policy. Compare collected fees less credit losses and incremental funding, servicing, and collection costs per eligible borrower. Track DPD30 alongside the financial result.

**Assumptions.** Group loans by disbursal month. A loan is mature if its due date plus 30 days is on or before the November 5 extract. Treat repayment within EGP 0.01 of principal plus fee as full repayment, using the last recorded payment date as a proxy for when the loan was fully repaid. Full repayment on the checkpoint date avoids DPD30 default. Count fees on fully repaid loans and estimate credit loss on defaulted loans as principal less repayments recorded at extract, floored at zero.

In [1]:
# Task 3.4 | Credit risk and the proposed limit cut
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine

ROOT = next(
    folder for folder in (Path.cwd(), Path.cwd().parent)
    if (folder / "data" / "cleaned").is_dir()
)
CLEANED = ROOT / "data" / "cleaned"
Q3_START = "2026-07-01"
Q3_END = "2026-10-01"
EXTRACT_DATE = "2026-11-05"

CHANNEL_MAP = {
    "meta": "Meta", "facebook ads": "Meta", "facebook": "Meta", "fb": "Meta",
    "instagram": "Meta", "ig": "Meta",
    "google": "Google Ads", "google ads": "Google Ads", "adwords": "Google Ads",
    "gads": "Google Ads", "tiktok": "TikTok", "tt": "TikTok",
    "organic": "Other", "(direct)": "Other", "referral": "Other",
    "ref": "Other", "invite": "Other",
}

loans = pd.read_csv(CLEANED / "loans.csv")
users = pd.read_csv(CLEANED / "users.csv", usecols=["user_id", "utm_source_normalized"])
users["channel"] = users["utm_source_normalized"].map(CHANNEL_MAP).fillna("Unknown")
settings = pd.DataFrame([{
    "q3_start": Q3_START, "q3_end": Q3_END,
    "extract_date": EXTRACT_DATE,
}])

engine = create_engine("sqlite://")
loans.to_sql("loans", engine, index=False)
users[["user_id", "channel"]].to_sql("users", engine, index=False)
settings.to_sql("settings", engine, index=False)

%load_ext sql
%config SqlMagic.displaylimit = 20
%sql engine

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

In [2]:
%%sql
-- 1. Reconcile Finance's snapshot with mature DPD30 by disbursement month.
DROP VIEW IF EXISTS temp.q3_loan_outcomes;

CREATE TEMP VIEW q3_loan_outcomes AS
WITH loan_checkpoints AS (
    SELECT l.*, s.extract_date, DATE(l.due_date, '+30 days') AS dpd30_date,
           STRFTIME('%Y-%m', l.disbursed_date) AS disbursal_month,
           CAST(STRFTIME('%d', l.disbursed_date) AS INTEGER) AS disbursal_day,
           l.principal_egp + l.fee_egp AS amount_due
    FROM loans AS l CROSS JOIN settings AS s
    WHERE l.disbursed_date >= s.q3_start AND l.disbursed_date < s.q3_end
),
repayment_flags AS (
    SELECT *,
           dpd30_date <= extract_date AS is_mature,
           amount_repaid_egp < amount_due - 0.01 AS short_at_extract
    FROM loan_checkpoints
)
SELECT *,
       CASE WHEN is_mature AND (short_at_extract OR last_repayment_date > dpd30_date)
            THEN 1 ELSE 0 END AS default_at_dpd30
FROM repayment_flags;

SELECT disbursal_month, COUNT(*) AS loans_disbursed,
       SUM(short_at_extract) AS loans_short_at_extract,
       ROUND(100.0 * SUM(short_at_extract) / COUNT(*), 2) AS short_at_extract_pct,
       SUM(is_mature) AS mature_loans,
       COUNT(*) - SUM(is_mature) AS immature_loans,
       SUM(default_at_dpd30) AS dpd30_defaults,
       ROUND(100.0 * SUM(default_at_dpd30) / NULLIF(SUM(is_mature), 0), 2) AS dpd30_pct
FROM q3_loan_outcomes
GROUP BY disbursal_month
ORDER BY disbursal_month;

Running query in 'sqlite://'

disbursal_month,loans_disbursed,loans_short_at_extract,short_at_extract_pct,mature_loans,immature_loans,dpd30_defaults,dpd30_pct
2026-07,260,17,6.54,260,0,17,6.54
2026-08,427,25,5.85,427,0,25,5.85
2026-09,623,73,11.72,353,270,34,9.63


In [3]:
%%sql
-- 2. Classify each loan once using the same checkpoints as the monthly rate.
WITH classified AS (
    SELECT disbursal_month,
           CASE
               WHEN NOT is_mature THEN 'Immature'
               WHEN short_at_extract THEN 'Default, still short'
               WHEN last_repayment_date IS NULL THEN 'Missing repayment date'
               WHEN last_repayment_date <= due_date THEN 'Paid on time'
               WHEN last_repayment_date <= dpd30_date THEN 'Paid late by DPD30'
               ELSE 'Default, later paid'
           END AS repayment_outcome
    FROM q3_loan_outcomes
)
SELECT disbursal_month, COUNT(*) AS loans_disbursed,
       SUM(repayment_outcome = 'Paid on time') AS paid_on_time,
       SUM(repayment_outcome = 'Paid late by DPD30') AS paid_late_by_dpd30,
       SUM(repayment_outcome = 'Default, later paid') AS default_later_repaid,
       SUM(repayment_outcome = 'Default, still short') AS default_still_short,
       SUM(repayment_outcome = 'Immature') AS immature,
       SUM(repayment_outcome = 'Missing repayment date') AS missing_repayment_date
FROM classified
GROUP BY disbursal_month
ORDER BY disbursal_month;

Running query in 'sqlite://'

disbursal_month,loans_disbursed,paid_on_time,paid_late_by_dpd30,default_later_repaid,default_still_short,immature,missing_repayment_date
2026-07,260,176,67,0,17,0,0
2026-08,427,277,125,0,25,0,0
2026-09,623,205,114,0,34,270,0


## Appendix | Matched-period and borrower-mix checks

In [4]:
%%sql
-- A1. Compare the same days of each month, using September's maturity cutoff.
WITH comparable_window AS (
    SELECT CAST(STRFTIME('%d', DATE(extract_date, '-44 days')) AS INTEGER) AS last_day
    FROM settings
)
SELECT disbursal_month, last_day AS through_day, COUNT(*) AS mature_loans,
       SUM(default_at_dpd30) AS dpd30_defaults,
       ROUND(100.0 * AVG(default_at_dpd30), 2) AS dpd30_pct
FROM q3_loan_outcomes CROSS JOIN comparable_window
WHERE is_mature AND disbursal_day <= last_day
GROUP BY disbursal_month, last_day
ORDER BY disbursal_month;

Running query in 'sqlite://'

disbursal_month,through_day,mature_loans,dpd30_defaults,dpd30_pct
2026-07,22,139,8,5.76
2026-08,22,288,16,5.56
2026-09,22,353,34,9.63


In [5]:
%%sql
-- A2. Compare mature default rates by booked amount and acquisition channel.
WITH mature_loans AS (
    SELECT q.disbursal_month, q.principal_egp,
           COALESCE(u.channel, 'Unknown') AS channel, q.default_at_dpd30
    FROM q3_loan_outcomes AS q
    LEFT JOIN users AS u USING (user_id)
    WHERE q.is_mature
),
segments AS (
    SELECT 'Booked principal (EGP)' AS breakdown,
           CAST(principal_egp AS TEXT) AS segment, disbursal_month, default_at_dpd30
    FROM mature_loans
    UNION ALL
    SELECT 'Acquisition channel', channel, disbursal_month, default_at_dpd30
    FROM mature_loans
)
SELECT breakdown, segment,
       COUNT(*) FILTER (WHERE disbursal_month = '2026-07') AS jul_mature,
       SUM(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-07') AS jul_defaults,
       ROUND(100.0 * AVG(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-07'), 2) AS jul_pct,
       COUNT(*) FILTER (WHERE disbursal_month = '2026-08') AS aug_mature,
       SUM(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-08') AS aug_defaults,
       ROUND(100.0 * AVG(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-08'), 2) AS aug_pct,
       COUNT(*) FILTER (WHERE disbursal_month = '2026-09') AS sep_mature,
       SUM(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-09') AS sep_defaults,
       ROUND(100.0 * AVG(default_at_dpd30) FILTER (WHERE disbursal_month = '2026-09'), 2) AS sep_pct
FROM segments
GROUP BY breakdown, segment
ORDER BY breakdown DESC, CAST(segment AS INTEGER), segment;

Running query in 'sqlite://'

breakdown,segment,jul_mature,jul_defaults,jul_pct,aug_mature,aug_defaults,aug_pct,sep_mature,sep_defaults,sep_pct
Booked principal (EGP),500,63,0,0.0,116,4,3.45,100,11,11.0
Booked principal (EGP),1000,81,5,6.17,132,7,5.3,94,11,11.7
Booked principal (EGP),1500,56,5,8.93,78,10,12.82,82,9,10.98
Booked principal (EGP),2000,35,4,11.43,62,4,6.45,47,2,4.26
Booked principal (EGP),3000,25,3,12.0,39,0,0.0,30,1,3.33
Acquisition channel,Google Ads,81,2,2.47,142,5,3.52,90,6,6.67
Acquisition channel,Meta,52,5,9.62,107,16,14.95,111,20,18.02
Acquisition channel,Other,80,6,7.5,110,2,1.82,91,1,1.1
Acquisition channel,TikTok,16,3,18.75,21,0,0.0,34,6,17.65
Acquisition channel,Unknown,31,1,3.23,47,2,4.26,27,1,3.7


In [6]:
%%sql
-- A3. Compare Q3 mature loan outcomes and economics by acquisition channel.
WITH mature_loan_economics AS (
    SELECT COALESCE(u.channel, 'Unknown') AS channel,
           q.default_at_dpd30,
           CASE
               WHEN q.short_at_extract = 0 THEN q.fee_egp
               ELSE 0
           END AS collected_fee_egp,
           CASE
               WHEN q.default_at_dpd30 = 1
               THEN MAX(q.principal_egp - q.amount_repaid_egp, 0)
               ELSE 0
           END AS credit_loss_egp
    FROM q3_loan_outcomes AS q
    LEFT JOIN users AS u ON u.user_id = q.user_id
    WHERE q.is_mature = 1
)
SELECT channel,
       COUNT(*) AS mature_loans,
       SUM(default_at_dpd30) AS dpd30_defaults,
       ROUND(100.0 * AVG(default_at_dpd30), 2) AS dpd30_pct,
       ROUND(SUM(collected_fee_egp), 2) AS collected_fees_egp,
       ROUND(SUM(credit_loss_egp), 2) AS credit_loss_egp,
       ROUND(SUM(collected_fee_egp - credit_loss_egp), 2) AS fees_less_credit_loss_egp
FROM mature_loan_economics
GROUP BY channel
ORDER BY fees_less_credit_loss_egp;

Running query in 'sqlite://'

channel,mature_loans,dpd30_defaults,dpd30_pct,collected_fees_egp,credit_loss_egp,fees_less_credit_loss_egp
Meta,270,41,15.19,23800.0,46032.82,-22232.82
TikTok,71,9,12.68,6760.0,12228.45,-5468.45
Unknown,105,4,3.81,10800.0,5576.39,5223.61
Other,281,9,3.2,26640.0,12744.88,13895.12
Google Ads,313,13,4.15,31360.0,13248.86,18111.14
